In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers, models
from datasets import load_dataset
from huggingface_hub import HfApi

np.random.seed(42)
tf.random.set_seed(42)
print("\n===== HUGGING FACE HUB =====")

api = HfApi()

models_list = list(api.list_models(limit=5))

print("Available Hugging Face Models:")

for model in models_list:
    print(model.id)


print("\n===== LOADING HUGGING FACE DATASET =====")

dataset = load_dataset("uoft-cs/cifar10")

print(dataset)

print("Training Samples:", len(dataset["train"]))
print("Testing Samples:", len(dataset["test"]))

# Display sample image

image = dataset["train"][0]["img"]

plt.imshow(image)
plt.title("Hugging Face CIFAR-10 Image")
plt.axis("off")
plt.show()

print("\n===== MODEL CARD =====")

model_card = """
Model Name: CIFAR-10 CNN

Description:
Convolutional Neural Network for image classification.

Dataset:
CIFAR-10 Dataset.

Architecture:
1. Convolution Layer
2. Max Pooling Layer
3. Flatten Layer
4. Dense Layer
5. Softmax Output Layer

Task:
Image Classification.

Evaluation:
Accuracy and Loss.

Limitations:
Performance depends on training data and parameters.
"""

print(model_card)

print("\n===== NUMPY CONVOLUTION =====")

image = np.array([
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10],
    [11, 12, 13, 14, 15],
    [16, 17, 18, 19, 20],
    [21, 22, 23, 24, 25]
])

kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1]
])


def convolution(image, kernel):

    h, w = image.shape
    kh, kw = kernel.shape

    output_h = h - kh + 1
    output_w = w - kw + 1

    output = np.zeros((output_h, output_w))

    for i in range(output_h):
        for j in range(output_w):

            region = image[i:i+kh, j:j+kw]

            output[i, j] = np.sum(region * kernel)

    return output


conv_result = convolution(image, kernel)

print("Original Image:")
print(image)

print("\nKernel:")
print(kernel)

print("\nConvolution Result:")
print(conv_result)

plt.imshow(conv_result, cmap="gray")
plt.title("Convolution Output")
plt.colorbar()
plt.show()


print("\n===== MAX POOLING =====")

feature_map = np.array([
    [1, 3, 2, 4],
    [5, 6, 7, 8],
    [3, 2, 1, 4],
    [7, 8, 9, 6]
])


def max_pooling(image, pool_size=2, stride=2):

    h, w = image.shape

    output_h = (h - pool_size) // stride + 1
    output_w = (w - pool_size) // stride + 1

    output = np.zeros((output_h, output_w))

    for i in range(output_h):
        for j in range(output_w):

            region = image[
                i*stride:i*stride+pool_size,
                j*stride:j*stride+pool_size
            ]

            output[i, j] = np.max(region)

    return output


pool_result = max_pooling(feature_map)

print("Original Feature Map:")
print(feature_map)

print("\nMax Pooling Result:")
print(pool_result)

plt.subplot(1, 2, 1)
plt.imshow(feature_map, cmap="gray")
plt.title("Original Feature Map")

plt.subplot(1, 2, 2)
plt.imshow(pool_result, cmap="gray")
plt.title("Max Pooling Output")

plt.show()

print("\n===== KERNEL SIZE, STRIDE AND PADDING =====")


def convolution_parameters(image, kernel, stride=1, padding=0):

    if padding > 0:

        image = np.pad(
            image,
            ((padding, padding), (padding, padding)),
            mode="constant"
        )

    h, w = image.shape
    kh, kw = kernel.shape

    output_h = (h - kh) // stride + 1
    output_w = (w - kw) // stride + 1

    output = np.zeros((output_h, output_w))

    for i in range(output_h):
        for j in range(output_w):

            region = image[
                i*stride:i*stride+kh,
                j*stride:j*stride+kw
            ]

            output[i, j] = np.sum(region * kernel)

    return output


sample_image = np.array([
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10],
    [11, 12, 13, 14, 15],
    [16, 17, 18, 19, 20],
    [21, 22, 23, 24, 25]
])

sample_kernel = np.ones((3, 3))

# Stride 1, Padding 0

result1 = convolution_parameters(
    sample_image, sample_kernel, stride=1, padding=0
)

# Stride 2, Padding 0

result2 = convolution_parameters(
    sample_image, sample_kernel, stride=2, padding=0
)

# Stride 1, Padding 1

result3 = convolution_parameters(
    sample_image, sample_kernel, stride=1, padding=1
)

print("Stride 1, Padding 0:")
print(result1)

print("\nStride 2, Padding 0:")
print(result2)

print("\nStride 1, Padding 1:")
print(result3)

print("\nOutput Shapes:")

print("Result 1:", result1.shape)
print("Result 2:", result2.shape)
print("Result 3:", result3.shape)


print("\n===== CIFAR-10 DATASET =====")

(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

class_names = [
    "Airplane", "Automobile", "Bird", "Cat", "Deer",
    "Dog", "Frog", "Horse", "Ship", "Truck"
]


plt.figure(figsize=(10, 5))

for i in range(10):

    plt.subplot(2, 5, i+1)

    plt.imshow(X_train[i])

    plt.title(class_names[y_train[i][0]])

    plt.axis("off")

plt.suptitle("CIFAR-10 Sample Images")
plt.tight_layout()
plt.show()

X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

print("Dataset Normalization Completed")

print("\n===== CNN MODEL =====")

model = models.Sequential([

    layers.Input(shape=(32, 32, 3)),

    # First Convolution Layer
    layers.Conv2D(
        32,
        (3, 3),
        activation="relu"
    ),

    # First Max Pooling Layer
    layers.MaxPooling2D((2, 2)),

    # Second Convolution Layer
    layers.Conv2D(
        64,
        (3, 3),
        activation="relu"
    ),

    # Second Max Pooling Layer
    layers.MaxPooling2D((2, 2)),

    # Third Convolution Layer
    layers.Conv2D(
        64,
        (3, 3),
        activation="relu"
    ),

    # Flatten Layer
    layers.Flatten(),

    # Dense Layer
    layers.Dense(64, activation="relu"),

    # Output Layer
    layers.Dense(10, activation="softmax")

])

model.summary()

model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

print("CNN Model Compiled Successfully")

print("\n===== CNN TRAINING =====")

history = model.fit(

    X_train,
    y_train,

    epochs=10,

    batch_size=64,

    validation_data=(X_test, y_test)

)

print("\n===== MODEL EVALUATION =====")

test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test
)

print("Test Loss:", test_loss)

print("Test Accuracy:", test_accuracy * 100, "%")

plt.figure(figsize=(12, 5))

# Accuracy

plt.subplot(1, 2, 1)

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("CNN Accuracy")

plt.xlabel("Epochs")

plt.ylabel("Accuracy")

plt.legend()


# Loss

plt.subplot(1, 2, 2)

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title("CNN Loss")

plt.xlabel("Epochs")

plt.ylabel("Loss")

plt.legend()

plt.tight_layout()
plt.show()

print("\n===== IMAGE PREDICTION =====")

predictions = model.predict(X_test[:10])

for i in range(10):

    predicted_class = np.argmax(predictions[i])

    actual_class = y_test[i][0]

    print("\nImage:", i+1)

    print("Predicted:", class_names[predicted_class])

    print("Actual:", class_names[actual_class])

print("\n===================================")
print("EXPERIMENT COMPLETED SUCCESSFULLY")
print("===================================")

print("Hugging Face Hub explored.")
print("Dataset loading demonstrated.")
print("Model cards understood.")
print("Convolution implemented using NumPy.")
print("Max Pooling implemented using NumPy.")
print("Kernel Size, Stride and Padding explored.")
print("CNN trained on CIFAR-10 dataset.")

print("\nFinal Test Accuracy:",
      round(test_accuracy * 100, 2), "%")

print("\nCONCLUSION:")
print("A CNN model was successfully developed and")
print("evaluated using the CIFAR-10 image dataset.")
